### NYC DOT Marketing Mix Modeling -- Meridian Modeling

#### Overview

In this notebook, Google's Meridian Bayesian Marketing Mix Modeling (MMM) framework is used to investigate the relationship between NYC DOT road-safety advertising expenditures and weekly motor vehicle crashes.

The objective is to estimate individual advertising channel contributions while accounting for seasonality, long-term trends, holidays, and congestion pricing. The model incorporates adstock and saturation to capture potential advertising carryover effects and diminishing returns.

Model convergence is evaluated, predictive performance, posterior uncertainty, and estimated channel contributions to determine whether the results provide meaningful insights into advertising effectiveness.

Because the analysis uses observational expenditure data rather than direct advertising exposure measurements, the findings are interpreted as modeled associations rather than established causal effects.

#### Import libraries

In [1]:
# Core analysis libraries
import pandas as pd
import numpy as np
from pathlib import Path

# Show all columns
pd.set_option('display.max_columns', None)

#### File paths

In [2]:
# File paths
PROJECT_ROOT = Path.cwd().resolve().parent
SRC_DIR = PROJECT_ROOT / "src"
CLEAN_DATA_DIR = PROJECT_ROOT / "data/clean_data"
RAW_DATA_DIR = PROJECT_ROOT / "data/raw_data"
MODELS_DIR = PROJECT_ROOT / "models"
DEPLOYMENT_DIR = PROJECT_ROOT / "deployment"

#### Import data

In [3]:
model_data = pd.read_parquet(CLEAN_DATA_DIR / "03_dot_mmm_feature_engineering.parquet")
model_data.head()

,week_start,spend_safety_tv,spend_safety_radio,spend_safety_print,spend_safety_digital_social,spend_safety_ooh_other,crashes,injured,killed,t,is_holiday_week,post_congestion,season_sin,season_cos,t_scaled
0,2022-07-04,0.00,0.00,0.00,0.00,0.00,1919,1053,12,0,1,0,-0.040844,-0.999166,0.000000
1,2022-07-11,21920.83,3129.63,3209.92,0.00,18135.30,1994,1191,9,1,0,0,-0.160575,-0.987024,0.006452
2,2022-07-18,25574.30,3651.24,3744.90,0.00,21157.85,2086,1183,10,2,0,0,-0.277979,-0.960587,0.012903
3,2022-07-25,25574.30,3651.24,53479.90,5730.00,35958.85,2044,1020,5,3,0,0,-0.391358,-0.920239,0.019355
4,2022-08-01,25574.30,3651.24,60952.90,2996.45,36106.01,2113,1127,5,4,0,0,-0.499069,-0.866562,0.025806


In [4]:
model_data.shape

(156, 15)

#### Data types and nulls

In [5]:
# Data types and nulls
data_types_null_counts = pd.DataFrame()
data_types_null_counts["Data Type"] = model_data.dtypes
data_types_null_counts["Count of Nulls"] = model_data.isna().sum()
data_types_null_counts

,Data Type,Count of Nulls
week_start,datetime64[ns],0
spend_safety_tv,float64,0
spend_safety_radio,float64,0
spend_safety_print,float64,0
spend_safety_digital_social,float64,0
spend_safety_ooh_other,float64,0
crashes,int64,0
injured,int64,0
killed,int64,0
t,int64,0


#### Establish media and control columns

Identified five advertising channels and five control variables for modeling. The controls account for time trend, seasonality, holidays, and congestion pricing, which may influence crash frequency independently of advertising expenditures.

In [6]:
# Change format type for 'week_start'
model_data["week_start"] = model_data["week_start"].dt.strftime("%Y-%m-%d")

In [7]:
# Media columns
media_cols = [
    "spend_safety_tv", 
    "spend_safety_radio", 
    "spend_safety_print",
    "spend_safety_digital_social", 
    "spend_safety_ooh_other"
]

In [8]:
# Media channels
media_channels = [
    "tv", 
    "radio", 
    "print", 
    "digital_social", 
    "ooh_other"
]

In [9]:
# Control columns
control_cols = [
    "t_scaled", 
    "post_congestion", 
    "is_holiday_week", 
    "season_sin", 
    "season_cos"
]

#### Import model libraries

In [10]:
# Silence library warnings. These warnings flag intentional model settings
import os, warnings, logging
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")
logging.getLogger("tensorflow").setLevel(logging.ERROR)

In [11]:
# Import libraries
import tensorflow_probability.substrates.jax as tfp_jax
from meridian.data import data_frame_input_data_builder as data_builder
from meridian.model import model, spec, prior_distribution
from meridian.analysis import visualizer

#### Builder input data loader for model

Configured Meridian with weekly crashes as a non-revenue KPI, using advertising expenditures as media inputs and the selected control variables to account for other factors influencing crash frequency.

In [12]:
# Input builder data
input_data = (
    data_builder.DataFrameInputDataBuilder(
        kpi_type="non_revenue", 
        default_time_column="week_start"
    )
    .with_kpi(
        model_data, 
        kpi_col="crashes"
    )
    .with_controls(
        model_data, 
        control_cols=control_cols
    )
    .with_media(
        model_data, 
        media_cols=media_cols, 
        media_spend_cols=media_cols, 
        media_channels=media_channels
    )
    .build()
)

#### Model specs

Selected a zero-centered Normal(0, 1) prior to allow advertising expenditures to have either positive or negative associations with crashes, rather than assuming road-safety advertising would necessarily reduce crashes.

Used a random 10% holdout to distribute test observations across the study period rather than concentrating them in the period following the introduction of congestion pricing. However, this approach does not establish future forecasting performance.

Meridian incorporates adstock to account for potential delayed and carryover effects of advertising and saturation to capture diminishing returns as spending increases.

In [13]:
# Set for single geographic region
zero = tfp_jax.distributions.Deterministic(loc=np.float64(0.0))

# Prior assumptions of data, doesn't force positive coefficients
prior = prior_distribution.PriorDistribution(
    beta_m=tfp_jax.distributions.Normal(loc=np.float64(0.0), scale=np.float64(1.0)),
    tau_g_excl_baseline=zero, eta_m=zero, eta_rf=zero, eta_om=zero, eta_orf=zero, xi_c=zero, xi_n=zero,
)

# Create 10% holdout
holdout_id = np.random.default_rng(42).random(len(model_data)) < 0.10

# Define model spec
model_spec = spec.ModelSpec(
    prior=prior, # probability distributions
    media_prior_type="coefficient",
    media_effects_dist="normal", # for one geo
    holdout_id=holdout_id
)

In [14]:
# Verify holdout size
holdout_id.sum()

np.int64(15)

#### Fit model

Sampled the prior and posterior distributions using four MCMC chains to assess whether independent sampling paths produced consistent posterior distributions.

In [15]:
# Instantiate model
mmm = model.Meridian(
    input_data=input_data, 
    model_spec=model_spec
)

# Set number of samples
mmm.sample_prior(500)

# Sample from posterior
mmm.sample_posterior(
    n_chains=4, 
    n_adapt=500, 
    n_burnin=500, 
    n_keep=1000, 
    seed=42
)

#### Model evaluations

In [16]:
# Model diagnostics to see how well Bayesian model performed. Use kpi=True since non-revenue
model_diag = visualizer.ModelDiagnostics(mmm, use_kpi=True)

# Model fit to assess model's estimates vs. actual
model_fit = visualizer.ModelFit(mmm, use_kpi=True)

#### R-hat & Convergence

R-hat values near 1 indicated satisfactory convergence, suggesting that the four sampling chains were exploring consistent posterior distributions.

- beta_m: advertising coefficients
- alpha_m: how advertising effects persist over time (adstock)
- ec_m & slope_m: how advertising response changes as spending increases (saturation)
- gamma_c: relationships between the controls and kpi (control coefficients)
- beta_gm, gamma_gc, knot_values, mu_t, sigma: Unexplained values

In [17]:
model_diag.plot_rhat_boxplot()

alt.LayerChart(...)

#### Prior vs. Posterior Distributions

Compared prior and posterior distributions to evaluate how the observed data updated the model's parameter estimates. TV showed a positive association with crashes, while Print leaned negative. Radio, Digital/Social, and OOH/Other remained centered near zero, indicating uncertainty about the direction of their associations.

In [18]:
# Prior and posterior
model_diag.plot_prior_and_posterior_distribution(parameter="beta_m")

alt.FacetChart(...)

#### Train vs. holdout predictive accuracy
The model achieved an R² of 0.741 on training data and 0.856 on test data, with MAPE of 3.74% and 2.74%. The higher test score likely reflects the small, randomly selected holdout rather than better generalization. Because the controls capture trend and seasonality, these metrics do not show how much is attributable to advertising, and predictive accuracy does not establish reliable attribution of individual advertising effects.

In [19]:
diag_df = model_diag.predictive_accuracy_table()
diag_df.pivot(
    index="metric",
    columns="evaluation_set",
    values="value"
)

evaluation_set,All Data,Test,Train
metric,,,
MAPE,0.036443,0.027437,0.037401
R_Squared,0.751576,0.856247,0.740728
wMAPE,0.035750,0.027597,0.036611


#### Controls-Only Baseline

All three models use the same 15 randomly held-out weeks. The linear regressions use raw weekly spend and do not model adstock or saturation, so the comparison with Meridian is not like-for-like. Test scores exceed training scores for every model, which suggests the holdout weeks are easier to predict rather than indicating better generalization.

In [20]:
# Baseline model libraries
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_percentage_error

In [21]:
# With control columns only
y = model_data["crashes"]
X = model_data[control_cols]

reg = LinearRegression().fit(X[~holdout_id], y[~holdout_id])

for name, mask in [("Train", ~holdout_id), ("Test", holdout_id)]:
    pred = reg.predict(X[mask])
    print(name, "R²:", round(r2_score(y[mask], pred), 3),
          "MAPE:", round(mean_absolute_percentage_error(y[mask], pred), 4))

Train R²: 0.622 MAPE: 0.0469
Test R²: 0.711 MAPE: 0.0375


In [22]:
# With control columns and media columns
y = model_data["crashes"]
X = model_data[control_cols + media_cols]

reg = LinearRegression().fit(X[~holdout_id], y[~holdout_id])

for name, mask in [("Train", ~holdout_id), ("Test", holdout_id)]:
    pred = reg.predict(X[mask])
    print(name, "R²:", round(r2_score(y[mask], pred), 3),
          "MAPE:", round(mean_absolute_percentage_error(y[mask], pred), 4))

Train R²: 0.702 MAPE: 0.0417
Test R²: 0.787 MAPE: 0.03


#### Model Comparison

| Model | Train R² | Test R² | Train MAPE | Test MAPE |
|---|---:|---:|---:|---:|
| Controls only (linear regression) | 0.622 | 0.711 | 4.69% | 3.75% |
| Controls + media (linear regression) | 0.702 | 0.787 | 4.17% | 3.00% |
| Meridian (Bayesian MMM) | 0.741 | 0.856 | 3.74% | 2.74% |

Adding media to the controls raised test R² from 0.711 to 0.787, so media spend carries some predictive information. This does not show that advertising affects crashes, since spend may track campaign timing or respond to crash levels, and the 15-week holdout makes the size of the gain uncertain.

#### Actual vs. expected model fit

Answers the questions of how closely does the model fit the observed crash data and how does the estimated outcome change when advertising contributions are removed?

The model's expected crash counts followed much of the observed trends and seasonality. Expected and baseline estimates remained relatively close, suggesting a limited overall advertising contribution relative to total crash frequency.

In [23]:
model_fit.plot_model_fit()

alt.LayerChart(...)

#### Advertising Channel Contributions

TV had an estimated positive contribution of 6,778 crashes, with a 95% credible interval of +690 to +14,202. Because the interval excludes zero, the model suggests a positive association between TV spending and crashes, although this does not establish causality.

Print had the largest negative mean estimated contribution (−9,366 crashes). However, its 95% credible interval (−18,596 to +2,077) includes both negative and positive values, meaning the direction of its association remains uncertain.

Radio, Digital/Social, and OOH/Other also had credible intervals crossing zero, indicating uncertainty about their individual contributions.

Across all five channels, the model estimated a combined contribution of −713 crashes, with a 95% credible interval of −4,521 to +3,076. Because the interval includes zero, the model cannot confidently determine whether advertising spending is associated with an overall increase or decrease in crash frequency.

In [24]:
# Advertising channel contributions
media_summary = visualizer.MediaSummary(
    mmm,
    use_kpi=True,
    confidence_level=0.95
)

contributions = (
    media_summary.get_paid_summary_metrics()
    ["incremental_outcome"]
    .sel(distribution="posterior")
)

display(contributions.to_pandas())

metric,mean,median,ci_lo,ci_hi
channel,,,,
tv,6777.545710,6528.170496,690.017972,14201.691194
radio,-1949.413738,-1494.909734,-12126.959092,6103.178717
print,-9366.015133,-9917.629200,-18595.699857,2077.193838
digital_social,1549.491538,1825.149624,-4372.753482,5925.297847
ooh_other,2275.485987,2792.294175,-5700.893282,7178.394468
All Channels,-712.905636,-697.006078,-4521.127397,3075.871318




#### Prior Sensitivity Analysis

In separate sensitivity tests, I evaluated prior standard deviations of 0.5, 1, and 2 to determine how sensitive the posterior estimates were to my prior assumptions.

These tests were run interactively and are not included in this notebook.

Although individual channel contribution estimates changed, the overall directional conclusions remained broadly consistent. This reduced my concern that the findings were driven primarily by my original prior selection.

#### Model Interpretation and Limitations

The positive association between TV spending and crashes may reflect reverse causality, where NYC DOT increases advertising in response to elevated crash numbers. Therefore, these results do not necessarily suggest that increased TV advertising causes more crashes.

The dataset also contains advertising expenditures rather than actual exposure metrics such as impressions, reach, and frequency. Without these measurements, I cannot determine how much audience exposure each expenditure generated, limiting my ability to evaluate individual channel effectiveness.

Additionally, overlapping channel spending and unobserved external factors may make it difficult to isolate individual advertising contributions.

#### Conclusions and Practical Implications

Overall, the results were inconclusive and do not provide sufficient evidence to recommend reallocating NYC DOT's advertising budget.

However, the positive association between TV spending and crashes warrants further investigation, particularly whether advertising expenditures increase in response to elevated crash rates.

The project demonstrates how Bayesian MMM can estimate advertising contributions and quantify uncertainty, while highlighting the importance of interpreting observational results within their real-world context.

#### Additional Research Opportunities

Several improvements could strengthen future analysis:

Advertising exposure: Incorporate impressions, reach, and frequency to better measure actual advertising activity rather than relying exclusively on expenditures.

Campaign timing: Investigate whether advertising campaigns are launched or expanded in response to elevated crash rates.

Adstock and saturation: Examine uncertainty in carryover effects and diminishing returns.

External controls: Incorporate more detailed measures of traffic volume and congestion pricing.

Causal identification: Explore stronger causal inference methods before using estimated contributions to inform budget reallocations.